# Transfer Learning with ResNet50 on TF Flowers Dataset
This notebook runs a 5-stage depth sweep with multi-seed training on TF Flowers (5 classes):
`linear_probe`, `stage5`, `stage4+5`, `stage3+4+5`, and `full`.

Refactor notes:
- Imports are consolidated in one top cell.
- Cells are split by responsibility.
- Training sweep cell is kept for future runs and is explicitly guarded for validation-only passes.


In [ ]:
# Imports
import os
import json
import time
import random

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications.resnet50 import preprocess_input


In [ ]:
# Runtime check
print("TF version:", tf.__version__)
print("All physical devices:")
print(tf.config.list_physical_devices())

print("\nGPU devices:")
print(tf.config.list_physical_devices("GPU"))


In [ ]:
# Download flower data and set dataset path
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
tf.keras.utils.get_file(origin=dataset_url, fname="flower_photos", untar=True)

data_dir = os.path.expanduser("~/.keras/datasets/flower_photos/flower_photos")

print("Data folder:", data_dir)
print(os.listdir(data_dir))


In [ ]:
# Build train/validation datasets
img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)


In [ ]:
# Dataset metadata
class_names = train_ds.class_names
print("Class names:", class_names)
print("Num classes:", len(class_names))


In [ ]:
# Augmentation and stable input pipeline
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomZoom(0.2),
], name="data_augmentation")

# Small dataset: in-memory cache is usually fine. Set False if RAM is constrained.
CACHE_IN_MEMORY = True
if CACHE_IN_MEMORY:
    train_ds = train_ds.cache()
    val_ds = val_ds.cache()

# Do not preprocess in tf.data; preprocessing is handled in the model.
train_ds = train_ds.shuffle(1000, seed=123, reshuffle_each_iteration=True).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


In [ ]:
# Custom loss
@tf.keras.utils.register_keras_serializable()
class SmoothedSparseCategoricalCrossentropy(tf.keras.losses.Loss):
    def __init__(self, num_classes=5, label_smoothing=0.1, name="smoothed_sparse_categorical_crossentropy"):
        super().__init__(name=name)
        self.num_classes = num_classes
        self.label_smoothing = label_smoothing
        self.inner = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing)

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_true = tf.one_hot(y_true, depth=self.num_classes)
        return self.inner(y_true, y_pred)

    def get_config(self):
        return {
            "num_classes": self.num_classes,
            "label_smoothing": self.label_smoothing,
            "name": self.name,
        }


In [ ]:
# Stage configuration and unfreeze rules
STAGE_ORDER = ["linear_probe", "stage5", "stage4+5", "stage3+4+5", "full"]
STAGE_LRS = {
    "linear_probe": 1e-3,
    "stage5": 1e-4,
    "stage4+5": 3e-5,
    "stage3+4+5": 3e-5,
    "full": 1e-5,
}


def _is_unfrozen_conv_for_stage(layer_name: str, train_stage: str) -> bool:
    if train_stage == "linear_probe":
        return False
    if train_stage == "stage5":
        return layer_name.startswith("conv5_")
    if train_stage == "stage4+5":
        return layer_name.startswith("conv4_") or layer_name.startswith("conv5_")
    if train_stage == "stage3+4+5":
        return (
            layer_name.startswith("conv3_")
            or layer_name.startswith("conv4_")
            or layer_name.startswith("conv5_")
        )
    if train_stage == "full":
        return layer_name.startswith("conv")
    raise ValueError(f"Unknown train_stage: {train_stage}")


In [ ]:
# Model builder
def build_model(train_stage: str):
    if train_stage not in STAGE_ORDER:
        raise ValueError(f"Unknown train_stage: {train_stage}")

    base_model = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Freeze first, then selectively unfreeze conv layers by stage depth.
    for layer in base_model.layers:
        layer.trainable = False

    for layer in base_model.layers:
        if _is_unfrozen_conv_for_stage(layer.name, train_stage):
            layer.trainable = True

    # Keep BN frozen for small data regardless of stage.
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = preprocess_input(x)
    x = base_model(x, training=False)  # Force inference-mode BN statistics.
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(5, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=STAGE_LRS[train_stage],
            weight_decay=1e-5
        ),
        loss=SmoothedSparseCategoricalCrossentropy(num_classes=5, label_smoothing=0.1),
        metrics=["accuracy"]
    )
    return model


In [ ]:
# Training utilities
RESULTS_PATH = "results_depth_sweep.json"
os.makedirs("Plot", exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def count_trainable_params(model):
    return int(np.sum([np.prod(w.shape) for w in model.trainable_weights]))


def run_one(train_stage: str, seed: int, epochs: int = 15):
    set_seed(seed)
    tf.keras.backend.clear_session()

    model = build_model(train_stage=train_stage)
    params = count_trainable_params(model)

    ckpt_path = f"Plot/best_{train_stage.replace('+', '_')}_seed{seed}.keras"
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt_path, monitor="val_loss", save_best_only=True, mode="min"
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True, mode="min"
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=1, min_lr=1e-7
        ),
    ]

    t0 = time.time()
    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=0
    )
    t1 = time.time()

    history_raw = {
        k: [float(v) for v in values]
        for k, values in hist.history.items()
    }

    train_acc = [float(v) for v in hist.history["accuracy"]]
    val_acc = [float(v) for v in hist.history["val_accuracy"]]
    train_loss = [float(v) for v in hist.history["loss"]]
    val_loss = [float(v) for v in hist.history["val_loss"]]

    best_idx = int(np.argmax(val_acc))

    return {
        "stage": train_stage,
        "seed": int(seed),
        "learning_rate": float(STAGE_LRS[train_stage]),
        "best_val_acc": float(val_acc[best_idx]),
        "best_val_loss": float(np.min(val_loss)),
        "epoch_of_best": int(best_idx + 1),
        "final_train_acc": float(train_acc[-1]),
        "final_val_acc": float(val_acc[-1]),
        "train_time_sec": float(t1 - t0),
        "trainable_params": int(params),
        "epochs_ran": int(len(train_acc)),
        "ckpt": ckpt_path,
        "metrics_per_epoch": {
            "train_acc": train_acc,
            "val_acc": val_acc,
            "train_loss": train_loss,
            "val_loss": val_loss,
        },
        "history": history_raw,
    }


In [ ]:
# Run multi-seed training across all unfreeze stages, aggregate metrics, and save results to JSON.
seeds = [1, 2, 3]
settings = STAGE_ORDER.copy()

runs = []
for st in settings:
    for s in seeds:
        run_out = run_one(st, s, epochs=15)
        runs.append(run_out)
        print(
            f"stage={st:>12s} seed={s} "
            f"best_val_acc={run_out['best_val_acc']:.4f} "
            f"epoch_of_best={run_out['epoch_of_best']} "
            f"trainable_params={run_out['trainable_params']}"
        )

summary = {}
for st in settings:
    sub = [r for r in runs if r["stage"] == st]
    best_accs = np.array([r["best_val_acc"] for r in sub], dtype=np.float32)
    best_losses = np.array([r["best_val_loss"] for r in sub], dtype=np.float32)
    times = np.array([r["train_time_sec"] for r in sub], dtype=np.float32)
    trainable_params = np.array([r["trainable_params"] for r in sub], dtype=np.int64)

    summary[st] = {
        "best_val_acc_mean": float(np.mean(best_accs)),
        "best_val_acc_std": float(np.std(best_accs, ddof=1)),
        "best_val_loss_mean": float(np.mean(best_losses)),
        "best_val_loss_std": float(np.std(best_losses, ddof=1)),
        "train_time_sec_mean": float(np.mean(times)),
        "train_time_sec_std": float(np.std(times, ddof=1)),
        "trainable_params_mean": int(np.mean(trainable_params)),
    }

results = {
    "config": {
        "dataset": "tf_flowers",
        "num_classes": 5,
        "image_size": [224, 224],
        "batch_size": 32,
        "stages": settings,
        "stage_learning_rates": STAGE_LRS,
        "seeds": seeds,
        "epochs": 15,
        "bn_policy": "batchnorm_frozen_in_all_stages; base_model_called_with_training_false",
    },
    "runs": runs,
    "summary": summary,
}

with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved: {RESULTS_PATH}")
print(json.dumps(summary, indent=2))


In [ ]:
# JSON load and aggregation helpers (plotting only)
if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(
        f"{RESULTS_PATH} not found. Run the sweep cell in a full training pass before plotting."
    )

with open(RESULTS_PATH, "r") as f:
    results = json.load(f)

runs = results["runs"]
stage_order = results["config"]["stages"]
summary = results["summary"]


def aggregate_stage_accuracy(stage_name: str):
    stage_runs = [r for r in runs if r["stage"] == stage_name]
    train_curves = [r["metrics_per_epoch"]["train_acc"] for r in stage_runs]
    val_curves = [r["metrics_per_epoch"]["val_acc"] for r in stage_runs]

    min_len = min(len(c) for c in train_curves)
    train_arr = np.array([c[:min_len] for c in train_curves], dtype=np.float32)
    val_arr = np.array([c[:min_len] for c in val_curves], dtype=np.float32)

    if train_arr.shape[0] > 1:
        train_std = train_arr.std(axis=0, ddof=1)
        val_std = val_arr.std(axis=0, ddof=1)
    else:
        train_std = np.zeros(min_len, dtype=np.float32)
        val_std = np.zeros(min_len, dtype=np.float32)

    return {
        "epoch": np.arange(1, min_len + 1),
        "train_mean": train_arr.mean(axis=0),
        "train_std": train_std,
        "val_mean": val_arr.mean(axis=0),
        "val_std": val_std,
        "n_seeds": train_arr.shape[0],
    }


In [ ]:
# Plot A: per-stage accuracy curves (mean +/- std across seeds)
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
axes = axes.flatten()

for i, stage in enumerate(stage_order):
    stats = aggregate_stage_accuracy(stage)
    ax = axes[i]
    e = stats["epoch"]

    ax.plot(e, stats["train_mean"], label="train_acc", linewidth=2)
    ax.plot(e, stats["val_mean"], label="val_acc", linewidth=2)
    ax.fill_between(
        e,
        stats["train_mean"] - stats["train_std"],
        stats["train_mean"] + stats["train_std"],
        alpha=0.15,
    )
    ax.fill_between(
        e,
        stats["val_mean"] - stats["val_std"],
        stats["val_mean"] + stats["val_std"],
        alpha=0.15,
    )

    ax.set_title(stage)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.grid(alpha=0.3)

if len(stage_order) < len(axes):
    for j in range(len(stage_order), len(axes)):
        axes[j].axis("off")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
fig.suptitle("TF Flowers ResNet50: Accuracy Curves Per Stage (mean +/- std across seeds)", y=1.02)
fig.tight_layout()
plt.show()


In [ ]:
# Plot B: depth vs best validation accuracy (mean +/- std across seeds)
means = np.array([summary[s]["best_val_acc_mean"] for s in stage_order], dtype=np.float32)
stds = np.array([summary[s]["best_val_acc_std"] for s in stage_order], dtype=np.float32)
x = np.arange(len(stage_order))

plt.figure(figsize=(10, 5))
plt.errorbar(x, means, yerr=stds, fmt="-o", capsize=5, linewidth=2)
plt.xticks(x, stage_order, rotation=20)
plt.ylabel("Best val accuracy")
plt.xlabel("Unfreeze depth stage")
plt.title("Depth Sweep: Best Validation Accuracy (mean +/- std across seeds)")
plt.grid(alpha=0.3)

for i, mean_val in enumerate(means):
    plt.text(i, mean_val + 0.002, f"{mean_val:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()
